[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/integrations/amazon_bedrock_agents_tracing_and_evals.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 Amazon Bedrock 智能体</h1>

本教程将：

- 构建 Amazon Bedrock 智能体
- 使用 Phoenix 为智能体添加埋点并追踪
- 为智能体追踪添加评估

## 背景

[Amazon Bedrock Agents](https://aws.amazon.com/bedrock/agents/) 是 Amazon Bedrock 中的全托管能力，可构建通过企业系统、数据源和 API 完成任务的 AI 智能体。智能体能够理解自然语言请求、将复杂任务拆解为步骤、检索相关信息并执行操作。可以据此创建回答问题、提供建议并代表用户操作的对话助手，同时在整段对话中保持上下文。

本教程使用 Phoenix 追踪与评估 Amazon Bedrock 智能体，展示它如何处理请求、做出决策，以及与不同系统交互来完成任务。

开始吧！

ℹ️ 本 Notebook 需要具备 Bedrock 访问权限的 AWS 账号。

### 安装依赖

In [ ]:
!pip install -q uv
!uv pip install -q arize-phoenix-otel boto3 anthropic openinference-instrumentation-bedrock

In [ ]:
import os
import time
from getpass import getpass

import boto3
import nest_asyncio
from phoenix.otel import register

nest_asyncio.apply()

### 设置 Phoenix 环境变量

本示例使用免费的在线托管版本 [Phoenix Cloud](https://app.phoenix.arize.com)，也可以选择[自托管 Phoenix](https://arize.com/docs/phoenix/self-hosting)。

In [ ]:
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
if not os.environ.get("PHOENIX_CLIENT_HEADERS"):
    os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + getpass("请输入 Phoenix API 密钥： ")

### 连接 Phoenix
现在将 Notebook 连接到 Phoenix 实例。

下方的 `auto_instrument` 标志会在环境中查找并调用所有已安装的 openinference-instrumentation 包。由于已经安装 openinference-instrumentation-bedrock，所有 Bedrock 和 Bedrock 智能体调用都将自动添加埋点并发送到 Phoenix。

In [ ]:
project_name = "Amazon Bedrock Agent Example"

tracer_provider = register(project_name=project_name, auto_instrument=True)

## 在 Bedrock Agents 中配置智能体

在 [Bedrock Agents](https://us-east-2.console.aws.amazon.com/bedrock/home?region=us-east-2#/overview) 中创建并配置智能体。本例使用：
1. 使用网页抓取工具创建的知识库。
2. 一组检索 Phoenix 信息的操作组函数。

Bedrock Agents 还支持 Guardrails、Prompts 等，Phoenix 也会追踪这些功能。

## 将 Notebook 连接到 AWS

接下来创建可连接 Bedrock 智能体的 AWS SSO 配置文件，可在 CLI 中使用 `aws configure sso`。完成配置并创建智能体后，填写下方变量：

In [ ]:
# SSO 配置
PROFILE_NAME = "phoenix"  # 已创建的 AWS SSO 配置名称
REGION = "us-east-2"  # Bedrock 智能体部署的区域
SERVICE_NAME = "bedrock-agent-runtime"  # Bedrock 智能体的服务名称

# Bedrock 智能体配置
AGENT_ID = ""  # Bedrock 智能体 ID，可在 Bedrock Agents 控制台查看
AGENT_ALIAS_ID = ""  # Bedrock 智能体别名 ID，可在 Bedrock Agents 控制台查看

In [ ]:
session = boto3.Session(profile_name=PROFILE_NAME)
bedrock_agent_runtime = session.client(SERVICE_NAME, region_name=REGION)

## 运行智能体
现在可以运行 Bedrock 智能体。

In [ ]:
def run(input_text):
    session_id = f"default-session1_{int(time.time())}"

    attributes = dict(
        inputText=input_text,
        agentId=AGENT_ID,
        agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id,
        enableTrace=True,
    )
    response = bedrock_agent_runtime.invoke_agent(**attributes)

    # 流式传输回复
    for _, event in enumerate(response["completion"]):
        if "chunk" in event:
            print(event)
            chunk_data = event["chunk"]
            if "bytes" in chunk_data:
                output_text = chunk_data["bytes"].decode("utf8")
                print(output_text)
        elif "trace" in event:
            print(event["trace"])

In [ ]:
run("Tell me about my recent Phoenix traces")

In [ ]:
run("How do I run evaluations in Arize Phoenix?")

In [ ]:
run("Tell me about my recent Phoenix experiments")

## 在 Phoenix 中查看追踪

现在应能在 Phoenix 控制台中看到追踪：

![Phoenix Bedrock 智能体追踪 1](https://storage.googleapis.com/arize-phoenix-assets/assets/images/bedrock-agent-traces-1.png)
![Phoenix Bedrock 智能体追踪 2](https://storage.googleapis.com/arize-phoenix-assets/assets/images/bedrock-agent-traces-2.png)

# 评估智能体

Phoenix 还内置 LLM 评估和基于代码的实验测试。下一节为追踪添加智能体工具调用评估。

此前使用的是较轻量的 Phoenix OTEL 追踪库。运行评估需要安装完整库。

In [ ]:
!pip install -q arize-phoenix

In [ ]:
import json

from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery
from phoenix.evals import (
    LLM,
    async_evaluate_dataframe,
)
from phoenix.evals.metrics import ToolInvocationEvaluator, ToolSelectionEvaluator

In [ ]:
query = (
    SpanQuery()
    .where(
        # 筛选 `LLM` 类型跨度。
        # 筛选条件是有效 Python 布尔表达式字符串。
        "span_kind == 'LLM'",
    )
    .select(
        question="input.value",
        outputs="output.value",
    )
)
trace_df = Client().spans.get_spans_dataframe(query=query, project_name=project_name)

In [ ]:
# 对 trace_df.input.value 的每行进行 JSON 解析
trace_df["question"] = trace_df["question"].apply(
    lambda x: json.loads(x).get("messages", [{}])[0].get("content", "") if isinstance(x, str) else x
)

In [ ]:
# 从输出提取工具调用（名称与参数）的函数
def extract_tool_calls(output_value):
    tool_calls = []
    try:
        o = json.loads(output_value)

        # 检查输出是否有表示消息组件列表的 'content'
        if "content" in o and isinstance(o["content"], list):
            for item in o["content"]:
                # 检查该项是否为 tool_use 类型
                if isinstance(item, dict) and item.get("type") == "tool_use":
                    # 提取所调用工具的名称与输入
                    tool_name = item.get("name")
                    tool_input = item.get("input", {})
                    if tool_name:
                        tool_calls.append(f"{tool_name}({json.dumps(tool_input)})")
    except (json.JSONDecodeError, TypeError, AttributeError):
        pass

    return tool_calls


# 对 trace_df.output.value 的每行应用此函数
trace_df["tool_call"] = trace_df["outputs"].apply(
    lambda x: extract_tool_calls(x) if isinstance(x, str) else []
)

# 显示找到的工具调用
print("追踪中找到的工具调用：", trace_df["tool_call"].sum())

In [ ]:
# 仅保留 tool_calls 非空的行（至少一次工具调用）
trace_df = trace_df[trace_df["tool_call"].apply(lambda x: len(x) > 0)]

trace_df.head()

In [ ]:
trace_df["tool_definitions"] = (
    "phoenix-traces 从 Phoenix 获取最新追踪信息，phoenix-experiments 获取最新实验信息，phoenix-datasets 获取最新数据集信息"
)

In [ ]:
from phoenix.evals import bind_evaluator

eval_model = LLM(provider="bedrock", model="anthropic.claude-3-5-haiku-20241022-v1:0")

# 将数据表的列映射到评估器预期字段
input_mapping = {
    "input": "question",
    "available_tools": "tool_definitions",
    "tool_selection": "tool_call",
}

tool_selection_evaluator = bind_evaluator(
    evaluator=ToolSelectionEvaluator(llm=eval_model),
    input_mapping=input_mapping,
)

tool_invocation_evaluator = bind_evaluator(
    evaluator=ToolInvocationEvaluator(llm=eval_model),
    input_mapping=input_mapping,
)

response_classifications = await async_evaluate_dataframe(
    dataframe=trace_df,
    evaluators=[tool_selection_evaluator, tool_invocation_evaluator],
    concurrency=10,
)
response_classifications["score"] = response_classifications.apply(
    lambda x: 1 if (x.get("tool_selection_score") or {}).get("label") == "correct" else 0,
    axis=1,
)

In [ ]:
from phoenix.client import AsyncClient
from phoenix.evals.utils import to_annotation_dataframe

annotations = to_annotation_dataframe(response_classifications)

px_client = AsyncClient()
await px_client.spans.log_span_annotations_dataframe(
    dataframe=annotations,
)

现在应能在 Phoenix 中看到评估标签！

![Bedrock 智能体评估 1](https://storage.googleapis.com/arize-phoenix-assets/assets/images/bedrock-agent-evals-1.png)
![Bedrock 智能体评估 2](https://storage.googleapis.com/arize-phoenix-assets/assets/images/bedrock-agent-evals-2.png)

# 后续步骤

可以在 Bedrock 中接入自己的工具和 Lambda 函数，扩展智能体能力；也可以了解 [Experiments](https://arize.com/docs/phoenix/datasets-and-experiments/how-to-experiments/run-experiments) 或 [Prompts](https://arize.com/docs/phoenix/prompt-engineering/overview-prompts)，扩展 Phoenix 中的测试和实验流程。

有关[智能体](https://arize.com/ai-agents/)和[评估](https://arize.com/llm-evaluation)的更多信息，请访问 Arize [网站](https://arize.com)。

期待看到你的应用！